In [ ]:
import pandas as pd
import numpy as np 
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import pickle 
import cv2
from tqdm import tqdm 
from skimage.color import label2rgb
import scanpy as sc
import anndata

In [ ]:
import igraph

In [ ]:
import distinctipy
def create_color_dict(labels_unique, colors_to_exclude=None):
    color_arr = distinctipy.get_colors(len(labels_unique), colors_to_exclude)
    cell_type_color_dict = {}
    for i in range(len(labels_unique)):
        cell_type_color_dict[labels_unique[i]] = color_arr[i]
    return cell_type_color_dict

In [ ]:
cluster_type = 'kmeans'

In [ ]:
key = "" # sample identifier

In [ ]:
registration_base_path = r"" # path to registered images

In [ ]:
pixel_clustering_base_path = r"" # path to pixel clustering results

In [ ]:
cell_cluster_base_path = r"" # path to cell phenotyping results

In [ ]:
pixel_barcode_base_path = r"" # path to cell pixel barcode results

In [ ]:
with open(pixel_clustering_base_path + '/wsi_labeled.pickle', 'rb') as f: 
    wsi_labeled = pickle.load(f)

In [ ]:
pixels_outside_cells = pd.read_csv(pixel_barcode_base_path + '/pixel_proportions_outside_cells.csv').drop(columns=['Unnamed: 0'])

In [ ]:
pixels_cells_barcode = pixels_outside_cells

In [ ]:
with open(cell_cluster_base_path + '/1x/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f: 
    cell_segmentation_mask = pickle.load(f)

In [ ]:
with open(registration_base_path + '/1x/roi_coord_dict.pickle', 'rb') as f: 
    roi = pickle.load(f)

In [ ]:
with open(pixel_clustering_base_path + '/cluster_color_dict.pickle', 'rb') as f:
    pixel_cluster_color_dict = pickle.load(f)

In [ ]:
cell_segmentation_mask_binary = np.zeros(cell_segmentation_mask.shape)
cell_segmentation_mask_binary[cell_segmentation_mask!=0] = 1
cell_segmentation_mask_binary = cv2.dilate(cv2.erode(cell_segmentation_mask_binary, np.ones((7,7)), np.uint8), iterations=1), np.ones((41, 41), np.uint8), iterations=1)

contours, _ = cv2.findContours(cell_segmentation_mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour = max(contours, key=cv2.contourArea)

contour_drawn = np.zeros((cell_segmentation_mask_binary.shape[0], cell_segmentation_mask_binary.shape[1]))
contour_drawn = cv2.drawContours(contour_drawn, [contour], -1, (255, 255, 255), -1)


In [ ]:
#cluster within subtype 

cluster_cost = {}
cluster_number = np.arange(1, 31)
for cell_type in list(pixels_cells_barcode['Cell Type'].unique()):
    if(cell_type !='Unclassified'):
        pixels_cells_subtype = pixels_cells_barcode[pixels_cells_barcode['Cell Type'] == cell_type]
        pixels_cells_subtype_arr = np.array(pixels_cells_subtype.iloc[:, :len(list(pixels_cells_barcode.columns))-4])
        cluster_cost[cell_type] = []
        for i in cluster_number: 
            kmeans = KMeans(n_clusters=i, random_state=42).fit(pixels_cells_subtype_arr)
            cluster_cost[cell_type].append(kmeans.inertia_)


In [ ]:
def find_elbow(k_values, wcss):
  
    k = np.array(k_values)
    y = np.array(wcss)
    A = np.array([k[0], y[0]])
    B = np.array([k[-1], y[-1]])
    AB = B - A
    distances = np.abs(np.cross(AB, A - np.column_stack((k, y)))) / np.linalg.norm(AB)
    idx = np.argmax(distances)
    return idx


In [ ]:
fig, ax = plt.subplots(1, len(list(pixels_cells_barcode['Cell Type'].unique()))-1)
fig.set_figheight(10)
fig.set_figwidth(50)
i = 0
for name, cost in cluster_cost.items():
    print(name)
    ax[i].plot(cluster_number, cost)
    for x, y in zip(cluster_number, cost): 
        ax[i].annotate(str(x), (x, y), 
                     textcoords="offset points",
                     xytext=(0, 0),
                     ha='center') 
    ax[i].plot([1, 30], [cost[0], cost[-1]], color='r')
    print(find_elbow(cluster_number, cost))

    ax[i].set_xlabel('Number of Clusters')
    ax[i].set_ylabel('WCSS')
    i+=1

In [ ]:
optimal_cluster_numbers = [4, 4, 4, 4]

In [ ]:
i = 0
cell_subtype_clusters = {}
cell_subtype_cluster_centers = {}
for cell_type in list(pixels_cells_barcode['Cell Type'].unique()):
    if(cell_type != 'Unclassified'):
        pixels_cells_subtype = pixels_cells_barcode[pixels_cells_barcode['Cell Type'] == cell_type]
        pixels_cells_subtype_arr = np.array(pixels_cells_subtype.iloc[:, :len(list(pixels_cells_barcode.columns))-4])
   
        kmeans = KMeans(n_clusters=optimal_cluster_numbers[i], random_state=42).fit(pixels_cells_subtype_arr)
        cell_subtype_clusters[cell_type] = kmeans.labels_
        cell_subtype_cluster_centers[cell_type] = kmeans.cluster_centers_
        i+=1

In [ ]:
with open(pixel_barcode_base_path + '/cell_subtype_cluster_centers_' + cluster_type + '.pickle', 'wb') as f: 
    pickle.dump(cell_subtype_cluster_centers, f)

In [ ]:
with open(pixel_barcode_base_path + '/cell_subtype_clusters_' + cluster_type + '.pickle', 'wb') as f: 
    pickle.dump(cell_subtype_clusters, f)

In [ ]:
cluster_color_dict_map = {}
colors_to_exclude = [(0, 0, 0), (1, 1, 1)]
for cell_type in list(pixels_cells_barcode['Cell Type'].unique()):
    if(cell_type != 'Unclassified'):
        cluster_color_dict_map[cell_type] = create_color_dict(list(set(cell_subtype_clusters[cell_type])), colors_to_exclude)
        colors_to_exclude.extend(list(cluster_color_dict_map[cell_type].values()))

In [ ]:
def create_image_labeled_with_cluster(cell_segmentation_mask, cell_id_list, cluster_assignments, cluster_color_dict):
    h, w = cell_segmentation_mask.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)

    cell_ids = np.array(cell_id_list)
    cluster_assignments = np.array(cluster_assignments)

    colors = np.array([
        (np.array(cluster_color_dict[cluster]) * 255).astype(np.uint8)
        for cluster in cluster_assignments
    ])

    
    flat_mask = cell_segmentation_mask.ravel()
    flat_out = out.reshape(-1, 3)
    max_id = flat_mask.max()
    lookup = np.zeros((max_id + 1, 3), dtype=np.uint8)
    lookup[cell_ids] = colors


    flat_out[:] = lookup[flat_mask]

    return out

image_labeled_dict = {}

for cell_type in pixels_cells_barcode['Cell Type'].unique():
    print(cell_type)
    if cell_type != 'Unclassified':
        pixels_cells_subtype = pixels_cells_barcode[pixels_cells_barcode['Cell Type'] == cell_type]
        cell_index = list(pixels_cells_subtype['Cell ID'])
        cluster_assignments = cell_subtype_clusters[cell_type]

        image_labeled = create_image_labeled_with_cluster(
            cell_segmentation_mask,
            cell_index,
            cluster_assignments,
            cluster_color_dict_map[cell_type]
        )

        image_labeled_dict[cell_type] = image_labeled


In [ ]:
from pathlib import Path

In [ ]:
with open(pixel_barcode_base_path + '/image_labeled_dict_pixels_outside_cells_' + cluster_type + '.pickle', 'wb') as f: 
    pickle.dump(image_labeled_dict, f)

In [ ]:
with open(pixel_barcode_base_path + '/image_labeled_dict_pixels_outside_cells_' + cluster_type + '.pickle', 'rb') as f: 
    image_labeled_dict = pickle.load(f)

In [ ]:
for cell_type, img in image_labeled_dict.items(): 
    print(cell_type)
    image_is_zero = (img == 0).all(axis=2)  
    cell_ids = np.unique(cell_segmentation_mask)
    cell_ids = cell_ids[cell_ids != 0]
    present_mask = (~image_is_zero) * cell_segmentation_mask 

    max_id = cell_ids.max()
    counts = np.bincount(present_mask.ravel(), minlength=max_id + 1)

    drawn_mask = np.zeros(max_id + 1, dtype=bool)
    drawn_mask[:len(counts)] = counts > 0  
    missing_pixels = ~drawn_mask[cell_segmentation_mask]  

    gray = np.array([128, 128, 128], dtype=img.dtype)
    img[missing_pixels] = gray

    img[contour_drawn == 0] = 255

In [ ]:
for cell_type, img in image_labeled_dict.items(): 
    image_is_zero = (img == 0).all(axis=2)
    img[image_is_zero] = 255
    image_labeled_dict[cell_type] = img


In [ ]:
with open(pixel_barcode_base_path + '/processed_image_labeled_dict_pixels_outside_cells_' + cluster_type + '.pickle', 'wb') as f: 
    pickle.dump(image_labeled_dict, f)

In [ ]:
fig, ax = plt.subplots(1, 4)
fig.set_figheight(25)
fig.set_figwidth(75)

i = 0
for cell_type in list(pixels_cells_barcode['Cell Type'].unique()):
    if(cell_type != 'Unclassified'):
        print(cell_type)
        ax[i].imshow(image_labeled_dict[cell_type])
        #wsi_scalebar = ScaleBar(0.1771*4, 'um', length_fraction=0.15, font_properties={'size':100})
        #ax.add_artist(wsi_scalebar)
        ax[i].axis('off')
        i+=1 
  


In [ ]:
fig, ax = plt.subplots(4, len(list(roi.keys())))
fig.set_figheight(30)
fig.set_figwidth(50)
i = 0

for cell_type in list(pixels_cells_barcode['Cell Type'].unique()):
    if(cell_type != 'Unclassified'):
        print(cell_type)
        j=0
        for name, coord in roi.items():
            ax[i, j].imshow(image_labeled_dict[cell_type][coord[0]:coord[1], coord[2]:coord[3]])
            ax[i, j].axis('off')

            j+=1 
        i+=1 

plt.subplots_adjust(wspace=0.025, hspace=0.025)

In [ ]:
def show_cell_region(image, mask, cell_id, radius=100):
  
    cell_pixels = np.where(mask == cell_id)
    cy = int(np.mean(cell_pixels[0]))
    cx = int(np.mean(cell_pixels[1]))
    y_min = max(cy - radius, 0)
    y_max = min(cy + radius, image.shape[0])
    x_min = max(cx - radius, 0)
    x_max = min(cx + radius, image.shape[1])

    cropped_img = image[y_min:y_max, x_min:x_max]
    
    plt.figure(figsize=(6,6))
    plt.imshow(cropped_img, cmap=plt.cm.colors.ListedColormap(pixel_cluster_color_dict.values()), vmin=1, vmax=9)
    plt.axis('off')
    plt.show()


In [ ]:
fig, ax = plt.subplots(4)
fig.set_figwidth(12)
fig.set_figheight(10)

scale = 4

max_clusters = max(optimal_cluster_numbers)

x_positions = np.linspace(0.05, 0.5, max_clusters)

i = 0

for cell_type in list(pixels_cells_barcode['Cell Type'].unique()):
    print(cell_type)

    if cell_type != 'Unclassified':

        num_clusters = optimal_cluster_numbers[i]

        for j in range(num_clusters):

            cluster_center = cell_subtype_cluster_centers[cell_type][j]
            cluster_center[np.isclose(cluster_center, 0, atol=1e-12)] = 0

            sizes = cluster_center
            colors = [pixel_cluster_color_dict[idx+1] for idx in range(len(cluster_center))]

            pie_width = 0.15 * scale
            pie_height = 0.15 * scale

            x_pos = x_positions[j]

            pie_ax = ax[i].inset_axes([
                x_pos - pie_width/2,
                0.5 - pie_height/2,
                pie_width,
                pie_height
            ])

            pie_ax.pie(
                sizes,
                colors=colors,
                startangle=90,
                wedgeprops={'edgecolor': 'black'}
            )

            pie_ax.set_aspect('equal')
            pie_ax.axis('off')

        ax[i].axis('off')
        i += 1

plt.tight_layout()


In [ ]:
with open(pixel_barcode_base_path + '/cluster_color_dict_map_' + cluster_type + '.pickle', 'wb') as f: 
    pickle.dump(cluster_color_dict_map, f)

In [ ]:
with open(pixel_barcode_base_path + '/cluster_color_dict_map_' + cluster_type + '.pickle', 'rb') as f: 
    cluster_color_dict_map = pickle.load(f)

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
fig, ax = plt.subplots(1)
cell_type = 'B-cells'
i = 0

cell_type_cluster_keys = list(cluster_color_dict_map[cell_type].keys())
cell_type_cluster_keys.sort()

cell_type_cluster_color_dict = {i: cluster_color_dict_map[cell_type][i] for i in cell_type_cluster_keys}

patches = [
    Line2D([], [], 
           color=color, 
           marker='o', 
           linestyle='None',
           markersize=8,
           label=label)
    for label, color in cell_type_cluster_color_dict.items()
]

ax.legend(
    patches,
    list(cell_type_cluster_color_dict.keys()),
    loc='center',
    frameon=False,
    ncol=len(cell_type_cluster_color_dict)/len(cell_type_cluster_color_dict), labelspacing=3
)

ax.axis('off')


In [ ]:
import matplotlib.patches as mpatches
fig, ax = plt.subplots(1)

i = 0
patches = [
    mpatches.Patch(color=color, label=label)
    for label, color in zip(list(pixel_cluster_color_dict.keys()), list(pixel_cluster_color_dict.values()))]
ax.legend(patches, list(pixel_cluster_color_dict.keys()), loc='center', frameon=False, ncol=3) #int(len(list(cluster_color_dict.keys()))))
ax.axis('off')
i+=1
plt.show()